# Simulation: Single ETF Buy & Hold

This notebook simulates the **Single ETF Buy & Hold Strategy**.

- **Portfolio:** 100% SPY, 0% QQQ, 0% VTI
- **Strategy:** Buy on day 1 and never sell. Add $500 every 10 trading days and buy more shares.
- **Initial Investment:** $10,000
- **Biweekly Contribution:** Add $500 every 10 trading days and buy more shares.

### How Buy & Hold Works
- Day 1: Invest $10,000 → buy SPY shares
- Day 10: Add $500 and buy more SPY shares
- Day 20: Add $500 and buy more SPY shares
- Day 30: Add $500 and buy more SPY shares
... 

---

## 1. Import Libraries, Configure Settings & Initialize Variables

In [11]:
import pandas as pd 
import os 

PROCESSED_DIR   = "../data/processed"
SIMULATIONS_DIR = "../data/simulations"
os.makedirs(SIMULATIONS_DIR, exist_ok=True)
INPUT_FILE  = os.path.join(PROCESSED_DIR, "prices_with_indicators.csv")
OUTPUT_FILE = os.path.join(SIMULATIONS_DIR, "single_buy_hold.csv")


initial_investment = 10000 # Starting investment on day 1
contribution_amount = 500 # Amount added every 10 trading days
contribution_interval = 10 # Every 10 trading days (biweekly)

# weight =  percentage of money going into each ETF expressed as a decimal
weight_spy = 1.0
weight_qqq = 0.0
weight_vti = 0.0

# Strategy/portfolio labels 
STRATEGY = "Buy & Hold"
PORTFOLIO_TYPE = "Single"

print(f"Input: {os.path.abspath(INPUT_FILE)}")
print(f"Output: {os.path.abspath(OUTPUT_FILE)}")
print(f"\nSimulation Parameters:")
print(f"Initial Investment: ${initial_investment:,}")
print(f"Contribution Amount: ${contribution_amount}")
print(f"Contribution Interval: Contributions are made every {contribution_interval} trading days")
print(f"Portfolio: SPY={weight_spy*100}%, QQQ={weight_qqq*100}%, VTI={weight_vti*100}%")
print(f"Strategy: {STRATEGY}")
print(f"Portfolio Type: {PORTFOLIO_TYPE}")

Input: c:\Users\Dell\Desktop\COMP3250-Project\data\processed\prices_with_indicators.csv
Output: c:\Users\Dell\Desktop\COMP3250-Project\data\simulations\single_buy_hold.csv

Simulation Parameters:
Initial Investment: $10,000
Contribution Amount: $500
Contribution Interval: Contributions are made every 10 trading days
Portfolio: SPY=100.0%, QQQ=0.0%, VTI=0.0%
Strategy: Buy & Hold
Portfolio Type: Single


## 2. Load `prices_with_indicators.csv`
Note:
- The first 49 rows have `NaN` in the MA50 columns — this is expected and correct.  
- Buy & Hold ignores MA50 entirely so these NaN values do not affect the simulation.

In [ ]:
df = pd.read_csv(INPUT_FILE, parse_dates=["Date"])

# Confirm the file loaded correctly
print(f"Loaded prices_with_indicators.csv")
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Nulls: {df.isnull().sum().to_dict()}")
print(f"\nFirst 5 rows:")
df.head()

Loaded prices_with_indicators.csv
Rows: 6217
Columns: ['Date', 'SPY_Price', 'QQQ_Price', 'VTI_Price', 'SPY_MA50', 'QQQ_MA50', 'VTI_MA50']
Date range: 2001-06-15 to 2026-03-06
Nulls: {'Date': 0, 'SPY_Price': 0, 'QQQ_Price': 0, 'VTI_Price': 0, 'SPY_MA50': 49, 'QQQ_MA50': 49, 'VTI_MA50': 49}

First 3 rows:


,Date,SPY_Price,QQQ_Price,VTI_Price,SPY_MA50,QQQ_MA50,VTI_MA50
0,2001-06-15,77.995598,35.981022,36.027668,NaN,NaN,NaN
1,2001-06-18,77.617950,35.499580,35.797894,NaN,NaN,NaN
2,2001-06-19,77.957207,35.322208,35.898216,NaN,NaN,NaN
3,2001-06-20,78.366867,36.124603,36.276844,NaN,NaN,NaN
4,2001-06-21,79.256599,36.614502,36.568089,NaN,NaN,NaN


## 3. Simulate Single ETF Buy & Hold Strategy

The simulation loops through each row through every trading day in the dataset.  

**Row 0: The Initial Investment:**
- cash = $10,000
- total_invested = $10,000
- shares_spy += cash / SPY_Price
- cash = 0

**Every row where i > 0 and i % 10 == 0 (Contribution Days):**
- contribution = $500
- cash += $500
- total_invested += $500
- shares_spy += cash / SPY_Price
- cash = 0

**All other rows:**
- contribution = 0
- cash = 0  
- shares_spy = same as previous row

Portfolio_Value = (shares_spy * SPY_Price) + cash

### For Single ETFBuy & Hold:
- `Position_SPY` is always 1 
- `Position_QQQ` and `Position_VTI` are always 0
- `Shares_QQQ` and `Shares_VTI` are always 0
- `Cash` is always **0** after shares are bought
- `SPY_MA50`, `QQQ_MA50`, `VTI_MA50` are left blank (NaN)

In [13]:
shares_spy = 0.0
cash = 0.0 
total_invested = 0.0  

results = []

for i, row in df.iterrows():
    spy_price = row["SPY_Price"]
    qqq_price = row["QQQ_Price"]
    vti_price = row["VTI_Price"]

    contribution = 0.0  
    
    # Initial Investment 
    if i == 0:
        cash = initial_investment
        total_invested = initial_investment

    elif i % contribution_interval == 0:
        contribution = contribution_amount
        cash += contribution
        total_invested += contribution


    if cash > 0:
        shares_spy += cash / spy_price
        cash = 0.0
        
    portfolio_value = (shares_spy * spy_price) + cash

    results.append({
        "Date": row["Date"],
        "SPY_Price": spy_price,
        "QQQ_Price": qqq_price,
        "VTI_Price": vti_price,
        "SPY_MA50": df.loc[i, "SPY_MA50"],  
        "QQQ_MA50": df.loc[i, "QQQ_MA50"],      
        "VTI_MA50": df.loc[i, "VTI_MA50"],     
        "Position_SPY": 1, 
        "Position_QQQ": 0,
        "Position_VTI": 0, 
        "Contribution": contribution,
        "Cash": cash,  
        "Shares_SPY": shares_spy,
        "Shares_QQQ": 0.0, 
        "Shares_VTI": 0.0,
        "Total_Invested": total_invested,
        "Portfolio_Value":portfolio_value,
        "Earnings": None,  
        "Daily_Return": None, 
        "Strategy": STRATEGY,
        "Portfolio_Type": PORTFOLIO_TYPE
    })

result_df = pd.DataFrame(results)
print(f"Simulation complete. Rows processed: {len(result_df)}")

Simulation complete. Rows processed: 6217


## 4. Calculate Earnings and Daily Return

- Earnings = Portfolio_Value - Total_Invested

- Daily_Return ( percentage change in portfolio value from the previous day)
- = Portfolio_Value.pct_change()
= (today - yesterday) / yesterday

- The first row of `Daily_Return` will always be `NaN` because there is no previous day to compare to.

In [14]:
result_df["Earnings"] = result_df["Portfolio_Value"] - result_df["Total_Invested"]
result_df["Daily_Return"] = result_df["Portfolio_Value"].pct_change()

print("Earnings and Daily Return calculated.")
print(result_df[["Date", "Portfolio_Value", "Total_Invested", "Earnings", "Daily_Return"]]
      .head(15).to_string(index=False))

Earnings and Daily Return calculated.
      Date  Portfolio_Value  Total_Invested    Earnings  Daily_Return
2001-06-15     10000.000000           10000    0.000000           NaN
2001-06-18      9951.580934           10000  -48.419066     -0.004842
2001-06-19      9995.077785           10000   -4.922215      0.004371
2001-06-20     10047.601305           10000   47.601305      0.005255
2001-06-21     10161.675995           10000  161.675995      0.011353
2001-06-22     10082.073442           10000   82.073442     -0.007834
2001-06-25      9989.331939           10000  -10.668061     -0.009199
2001-06-26      9975.380119           10000  -24.619881     -0.001397
2001-06-27      9969.637209           10000  -30.362791     -0.000576
2001-06-28     10024.621837           10000   24.621837      0.005515
2001-06-29     10561.551169           10500   61.551169      0.053561
2001-07-02     10693.352327           10500  193.352327      0.012479
2001-07-03     10690.767889           10500  190.767

## 5. Save `single_buy_hold.csv`

Save the simulation results to `data/simulations/`.  
The file will have exactly 21 columns as defined in the data dictionary.

In [15]:
final_cols = [
    "Date", "SPY_Price", "QQQ_Price", "VTI_Price",
    "SPY_MA50", "QQQ_MA50", "VTI_MA50",
    "Position_SPY", "Position_QQQ", "Position_VTI",
    "Contribution", "Cash",
    "Shares_SPY", "Shares_QQQ", "Shares_VTI",
    "Total_Invested", "Portfolio_Value", "Earnings",
    "Daily_Return", "Strategy", "Portfolio_Type"
]

result_df = result_df[final_cols]
result_df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved single_buy_hold.csv")
print(f"Path: {os.path.abspath(OUTPUT_FILE)}")
print(f"Rows: {len(result_df)}")
print(f"Columns: {len(result_df.columns)} —> {list(result_df.columns)}")
print(f"\nFinal Portfolio Summary")
print(f"Total Invested: ${result_df['Total_Invested'].iloc[-1]:,.2f}")
print(f"Final Value: ${result_df['Portfolio_Value'].iloc[-1]:,.2f}")
print(f"Total Earnings: ${result_df['Earnings'].iloc[-1]:,.2f}")

Saved single_buy_hold.csv
Path: c:\Users\Dell\Desktop\COMP3250-Project\data\simulations\single_buy_hold.csv
Rows: 6217
Columns: 21 —> ['Date', 'SPY_Price', 'QQQ_Price', 'VTI_Price', 'SPY_MA50', 'QQQ_MA50', 'VTI_MA50', 'Position_SPY', 'Position_QQQ', 'Position_VTI', 'Contribution', 'Cash', 'Shares_SPY', 'Shares_QQQ', 'Shares_VTI', 'Total_Invested', 'Portfolio_Value', 'Earnings', 'Daily_Return', 'Strategy', 'Portfolio_Type']

Final Portfolio Summary
Total Invested: $320,500.00
Final Value: $1,729,330.94
Total Earnings: $1,408,830.94
